# 04 — RAG ATM end-to-end

## Retrieval-Augmented Generation: unir recuperación + LLM

Ya hemos construido por separado:

```text
01_embeddings_semantic_search
    → retrieve(question)

03_llm_application_basics
    → ask_atm_llm(question, context)
```

Ahora los conectaremos:

```text
                 CORPUS ATM
                     │
              embeddings
                     │
                     ▼
PREGUNTA ─────→ retrieval
                     │
                     ▼
              TOP-K documentos
                     │
                     ▼
                  contexto
                     │
             pregunta + contexto
                     │
                     ▼
                    LLM
                     │
                     ▼
          respuesta con fuentes
```

Eso es un **RAG (Retrieval-Augmented Generation)** básico.

El objetivo es distinguir con claridad dos subsistemas diferentes:

1. **retrieval**: encontrar la evidencia adecuada;
2. **generation**: utilizar esa evidencia para responder.

## 0. Dependencias

Si vienes siguiendo los notebooks anteriores:

```bash
uv add sentence-transformers scikit-learn pandas openai python-dotenv
```

La API key debe estar fuera del notebook, en el archivo `.env` de la raíz del repositorio (ya está en `.gitignore`):

```text
OPENAI_API_KEY=sk-...
```

Opcionalmente, en el mismo `.env`:

```text
OPENAI_MODEL=gpt-6-luna
```

Usaremos `gpt-6-luna` por defecto para mantener el ejercicio ligero.

## 1. Importaciones y configuración

In [1]:
import os
import pandas as pd

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

load_dotenv()

pd.set_option("display.max_colwidth", 140)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "Falta OPENAI_API_KEY. Añádela al archivo .env en la raíz del repositorio."
    )

client = OpenAI()

print("Embedding model:", EMBEDDING_MODEL)
print("LLM model:", LLM_MODEL)

Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
LLM model: gpt-6-luna


# 2. Corpus ATM

En un sistema real estos fragmentos podrían proceder de manuales, procedimientos, especificaciones, incidencias o documentación técnica. Aquí utilizamos un corpus pequeño para poder inspeccionarlo visualmente.

In [2]:
documents = pd.DataFrame([
    {"id": "ATM-001", "area": "capacity",
     "text": "La capacidad del sector se ha reducido por un incremento inesperado del tráfico aéreo."},
    {"id": "ATM-002", "area": "weather",
     "text": "Las tormentas en la zona oeste están obligando a desviar varias rutas y aumentando las demoras."},
    {"id": "ATM-003", "area": "surveillance",
     "text": "Se ha detectado una degradación temporal en la cobertura del radar secundario."},
    {"id": "ATM-004", "area": "communications",
     "text": "La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad."},
    {"id": "ATM-005", "area": "staffing",
     "text": "La dotación disponible es inferior a la planificada y será necesario reducir la configuración del sector."},
    {"id": "ATM-006", "area": "flight_planning",
     "text": "El sistema de planificación de vuelos está procesando con retraso algunos planes de vuelo."},
    {"id": "ATM-007", "area": "capacity",
     "text": "La demanda prevista supera el número de aeronaves que el sector puede gestionar durante la próxima hora."},
    {"id": "ATM-008", "area": "weather",
     "text": "La presencia de cumulonimbos está generando desvíos y cambios frecuentes de nivel de vuelo."},
    {"id": "ATM-009", "area": "runway",
     "text": "Una de las pistas permanece temporalmente cerrada por inspección y la capacidad del aeropuerto se ha reducido."},
    {"id": "ATM-010", "area": "communications",
     "text": "Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales."},
    {"id": "ATM-011", "area": "flow_management",
     "text": "Se ha aplicado una regulación ATFM para limitar la entrada de tráfico en el sector congestionado."},
    {"id": "ATM-012", "area": "surveillance",
     "text": "La información de vigilancia presenta actualizaciones irregulares para algunas aeronaves."},
    {"id": "ATM-013", "area": "sector_configuration",
     "text": "Se ha abierto una configuración adicional de sectores para absorber el aumento de demanda."},
    {"id": "ATM-014", "area": "coordination",
     "text": "La carga de coordinación entre sectores adyacentes ha aumentado por múltiples reroutings."},
    {"id": "ATM-015", "area": "normal",
     "text": "Todos los sistemas operacionales funcionan con normalidad y el nivel de tráfico se mantiene estable."},
])

documents

,id,area,text
0,ATM-001,capacity,La capacidad del sector se ha reducido por un incremento inesperado del tráfico aéreo.
1,ATM-002,weather,Las tormentas en la zona oeste están obligando a desviar varias rutas y aumentando las demoras.
2,ATM-003,surveillance,Se ha detectado una degradación temporal en la cobertura del radar secundario.
3,ATM-004,communications,La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad.
4,ATM-005,staffing,La dotación disponible es inferior a la planificada y será necesario reducir la configuración del sector.
5,ATM-006,flight_planning,El sistema de planificación de vuelos está procesando con retraso algunos planes de vuelo.
6,ATM-007,capacity,La demanda prevista supera el número de aeronaves que el sector puede gestionar durante la próxima hora.
7,ATM-008,weather,La presencia de cumulonimbos está generando desvíos y cambios frecuentes de nivel de vuelo.
8,ATM-009,runway,Una de las pistas permanece temporalmente cerrada por inspección y la capacidad del aeropuerto se ha reducido.
9,ATM-010,communications,Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales.


# 3. Indexación: embeddings del corpus

Esta fase suele realizarse antes de que llegue la pregunta del usuario:

```text
documentos → chunking → embeddings → índice vectorial
```

Aquí cada fila ya es un fragmento y almacenaremos sus vectores en memoria.

In [3]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

document_embeddings = embedding_model.encode_document(
    documents["text"].tolist(),
    normalize_embeddings=True,
)

print("Embeddings shape:", document_embeddings.shape)

Embeddings shape: (15, 384)


# 4. Retrieval

In [4]:
def retrieve(question, top_k=3):
    query_embedding = embedding_model.encode_query(
        [question],
        normalize_embeddings=True,
    )

    scores = cosine_similarity(
        query_embedding,
        document_embeddings,
    )[0]

    results = documents.copy()
    results["similarity"] = scores

    return (
        results
        .sort_values("similarity", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [5]:
question = "¿Qué situaciones pueden reducir la capacidad disponible?"
retrieved = retrieve(question, top_k=5)
retrieved.round({"similarity": 3})

,id,area,text,similarity
0,ATM-005,staffing,La dotación disponible es inferior a la planificada y será necesario reducir la configuración del sector.,0.608
1,ATM-001,capacity,La capacidad del sector se ha reducido por un incremento inesperado del tráfico aéreo.,0.579
2,ATM-010,communications,Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales.,0.470
3,ATM-004,communications,La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad.,0.469
4,ATM-007,capacity,La demanda prevista supera el número de aeronaves que el sector puede gestionar durante la próxima hora.,0.436


Antes de llamar al LLM debemos detenernos aquí. Un RAG tiene **dos sistemas distintos**:

```text
RETRIEVAL  → ¿hemos encontrado la evidencia adecuada?
GENERATION → ¿ha utilizado bien el LLM esa evidencia?
```

Si retrieval falla, un excelente LLM recibirá información equivocada o incompleta.

# 5. Construcción del contexto

In [6]:
def build_context(retrieved_docs):
    parts = []

    for _, row in retrieved_docs.iterrows():
        parts.append(
            f"[{row['id']}] Área: {row['area']}\n{row['text']}"
        )

    return "\n\n".join(parts)

In [7]:
context = build_context(retrieved)
print(context)

[ATM-005] Área: staffing
La dotación disponible es inferior a la planificada y será necesario reducir la configuración del sector.

[ATM-001] Área: capacity
La capacidad del sector se ha reducido por un incremento inesperado del tráfico aéreo.

[ATM-010] Área: communications
Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales.

[ATM-004] Área: communications
La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad.

[ATM-007] Área: capacity
La demanda prevista supera el número de aeronaves que el sector puede gestionar durante la próxima hora.


Incluimos identificadores para que el modelo pueda señalar qué fuente respalda cada afirmación. Esto aporta trazabilidad, aunque una cita generada por el modelo también debe verificarse.

# 6. Generation: responder con la evidencia recuperada

In [8]:
RAG_INSTRUCTIONS = """
Eres un asistente técnico especializado en ATM.

Reglas:
1. Responde en español.
2. Usa exclusivamente la información del CONTEXTO.
3. No añadas hechos que no estén respaldados por el CONTEXTO.
4. Si el CONTEXTO no permite responder, dilo explícitamente.
5. Para cada afirmación factual relevante, indica la fuente usando su identificador entre corchetes, por ejemplo [ATM-007].
6. No cites una fuente que no aparezca en el CONTEXTO.
"""

def generate_answer(question, context):
    prompt = f"""
CONTEXTO:
{context}

PREGUNTA:
{question}
"""

    response = client.responses.create(
        model=LLM_MODEL,
        instructions=RAG_INSTRUCTIONS,
        input=prompt,
    )

    return response.output_text

In [9]:
answer = generate_answer(question, context)
print(answer)

Las situaciones respaldadas por el contexto son:

- Un incremento inesperado del tráfico aéreo, que ya ha reducido la capacidad del sector [ATM-001].
- Una dotación inferior a la planificada, que obliga a reducir la configuración del sector [ATM-005].

Además, la demanda prevista supera lo que el sector puede gestionar durante la próxima hora; esto indica que la capacidad disponible no alcanza para cubrir esa demanda, aunque el contexto no dice que la capacidad haya disminuido [ATM-007].

Se mencionan problemas de comunicaciones, pero el contexto no especifica que reduzcan la capacidad [ATM-004] [ATM-010].


# 7. Pipeline RAG completo

In [10]:
def rag(question, top_k=3, show_retrieval=False):
    retrieved_docs = retrieve(question, top_k=top_k)
    context = build_context(retrieved_docs)
    answer = generate_answer(question, context)

    if show_retrieval:
        return {
            "question": question,
            "retrieved": retrieved_docs,
            "context": context,
            "answer": answer,
        }

    return answer

In [11]:
result = rag(
    "¿Qué problemas de comunicaciones aparecen en la información disponible?",
    top_k=3,
    show_retrieval=True,
)

print("PREGUNTA")
print(result["question"])

print("\nDOCUMENTOS RECUPERADOS")
display(result["retrieved"].round({"similarity": 3}))

print("\nRESPUESTA")
print(result["answer"])

PREGUNTA
¿Qué problemas de comunicaciones aparecen en la información disponible?

DOCUMENTOS RECUPERADOS


,id,area,text,similarity
0,ATM-010,communications,Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales.,0.755
1,ATM-004,communications,La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad.,0.564
2,ATM-012,surveillance,La información de vigilancia presenta actualizaciones irregulares para algunas aeronaves.,0.393



RESPUESTA
Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales y pérdidas intermitentes de conectividad en la red de comunicaciones tierra-tierra. [ATM-010] [ATM-004]


Acabamos de construir nuestro primer RAG end-to-end:

```text
question → retrieve() → documents → build_context() → context → generate_answer() → answer
```

# 8. Pregunta cuya evidencia está repartida

In [12]:
question = "¿Qué factores relacionados con meteorología y tráfico pueden afectar a la operación?"

result = rag(question, top_k=5, show_retrieval=True)

display(
    result["retrieved"][["id", "area", "text", "similarity"]]
    .round({"similarity": 3})
)

print(result["answer"])

,id,area,text,similarity
0,ATM-002,weather,Las tormentas en la zona oeste están obligando a desviar varias rutas y aumentando las demoras.,0.487
1,ATM-008,weather,La presencia de cumulonimbos está generando desvíos y cambios frecuentes de nivel de vuelo.,0.428
2,ATM-001,capacity,La capacidad del sector se ha reducido por un incremento inesperado del tráfico aéreo.,0.379
3,ATM-011,flow_management,Se ha aplicado una regulación ATFM para limitar la entrada de tráfico en el sector congestionado.,0.373
4,ATM-012,surveillance,La información de vigilancia presenta actualizaciones irregulares para algunas aeronaves.,0.348


Los factores que pueden afectar a la operación son:

- **Meteorología:** las tormentas en la zona oeste obligan a desviar rutas y aumentan las demoras [ATM-002]. La presencia de cumulonimbos también provoca desvíos y cambios frecuentes de nivel de vuelo [ATM-008].
- **Tráfico:** un incremento inesperado del tráfico reduce la capacidad del sector [ATM-001]. Para limitar la entrada de tráfico en el sector congestionado, se ha aplicado una regulación ATFM [ATM-011].


Aquí aparece uno de los valores de RAG: retrieval encuentra piezas y el LLM puede **sintetizar información procedente de varios fragmentos**.

# 9. Pregunta sin respuesta

In [13]:
question = "¿Cuál es la frecuencia radio asignada al sector?"

result = rag(question, top_k=3, show_retrieval=True)

display(
    result["retrieved"][["id", "area", "text", "similarity"]]
    .round({"similarity": 3})
)

print(result["answer"])

,id,area,text,similarity
0,ATM-003,surveillance,Se ha detectado una degradación temporal en la cobertura del radar secundario.,0.360
1,ATM-004,communications,La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad.,0.334
2,ATM-011,flow_management,Se ha aplicado una regulación ATFM para limitar la entrada de tráfico en el sector congestionado.,0.311


El contexto no especifica la frecuencia radio asignada al sector, por lo que no se puede determinar. [ATM-003] [ATM-004] [ATM-011]


Esta prueba es crucial: retrieval **siempre devolverá algo**, aunque ningún documento contenga la respuesta.

`top_k=3` significa “devuelve los tres más similares”, no “devuelve tres documentos suficientemente relevantes”.

# 10. Umbral sencillo de retrieval

In [14]:
def retrieve_with_threshold(question, top_k=3, min_similarity=0.35):
    results = retrieve(question, top_k=top_k)

    return results[
        results["similarity"] >= min_similarity
    ].reset_index(drop=True)

**Advertencia:** `0.35` es solo un valor didáctico. No existe un threshold universal; debe calibrarse con datos de evaluación del dominio.

In [15]:
question = "¿Cuál es la frecuencia radio asignada al sector?"

retrieve_with_threshold(
    question,
    top_k=5,
    min_similarity=0.35,
).round({"similarity": 3})

,id,area,text,similarity
0,ATM-003,surveillance,Se ha detectado una degradación temporal en la cobertura del radar secundario.,0.36


In [17]:
def rag_with_threshold(question, top_k=3, min_similarity=0.35):
    retrieved_docs = retrieve_with_threshold(
        question,
        top_k=top_k,
        min_similarity=min_similarity,
    )

    if retrieved_docs.empty:
        return {
            "retrieved": retrieved_docs,
            "answer": "No se ha encontrado evidencia suficiente en el corpus para responder.",
        }

    context = build_context(retrieved_docs)
    answer = generate_answer(question, context)

    return {
        "retrieved": retrieved_docs,
        "answer": answer,
    }

# 11. Evaluar retrieval sin usar el LLM

In [18]:
retrieval_eval = [
    {
        "question": "¿Hay problemas de cobertura radar?",
        "expected_ids": {"ATM-003"},
    },
    {
        "question": "¿Qué ocurre con el intercambio de mensajes entre sistemas?",
        "expected_ids": {"ATM-010"},
    },
    {
        "question": "¿La falta de personal puede reducir la configuración?",
        "expected_ids": {"ATM-005"},
    },
    {
        "question": "¿Hay tormentas que obliguen a desviar rutas?",
        "expected_ids": {"ATM-002", "ATM-008"},
    },
    {
        "question": "¿Existe una regulación para limitar tráfico?",
        "expected_ids": {"ATM-011"},
    },
]

In [19]:
def evaluate_retrieval(eval_set, top_k=3):
    rows = []

    for item in eval_set:
        results = retrieve(item["question"], top_k=top_k)
        retrieved_ids = set(results["id"])
        hit = bool(item["expected_ids"] & retrieved_ids)

        rows.append({
            "question": item["question"],
            "expected": sorted(item["expected_ids"]),
            "retrieved": results["id"].tolist(),
            "hit": hit,
        })

    return pd.DataFrame(rows)

In [20]:
retrieval_results = evaluate_retrieval(retrieval_eval, top_k=3)
retrieval_results

,question,expected,retrieved,hit
0,¿Hay problemas de cobertura radar?,[ATM-003],"[ATM-003, ATM-009, ATM-012]",True
1,¿Qué ocurre con el intercambio de mensajes entre sistemas?,[ATM-010],"[ATM-010, ATM-004, ATM-014]",True
2,¿La falta de personal puede reducir la configuración?,[ATM-005],"[ATM-005, ATM-010, ATM-001]",True
3,¿Hay tormentas que obliguen a desviar rutas?,"[ATM-002, ATM-008]","[ATM-002, ATM-008, ATM-004]",True
4,¿Existe una regulación para limitar tráfico?,[ATM-011],"[ATM-011, ATM-001, ATM-004]",True


In [21]:
hit_rate = retrieval_results["hit"].mean()
print("Hit Rate@3:", round(float(hit_rate), 3))

Hit Rate@3: 1.0


**Hit Rate@K** responde a una pregunta muy sencilla:

> ¿aparece al menos un documento relevante dentro del top K?

Más adelante podríamos usar Recall@K, Precision@K, MRR, nDCG o evaluaciones específicas del negocio.

# 12. Dos tipos de fallo

### Retrieval correcto, generation incorrecta

El documento adecuado llega al LLM, pero el modelo puede interpretarlo mal, omitir condiciones, mezclar fuentes o añadir información.

### Generation correcta, retrieval insuficiente

El LLM responde perfectamente a lo recibido, pero faltaba una fuente esencial. En ese caso no se arregla cambiando el prompt: hay que mejorar retrieval.

# 13. `top_k` como hiperparámetro del sistema

In [22]:
question = "¿Qué puede reducir la capacidad?"

for top_k in [1, 2, 3, 5, 8]:
    results = retrieve(question, top_k=top_k)

    print("=" * 80)
    print("TOP_K =", top_k)
    display(
        results[["id", "area", "similarity"]]
        .round({"similarity": 3})
    )

TOP_K = 1


,id,area,similarity
0,ATM-005,staffing,0.586


TOP_K = 2


,id,area,similarity
0,ATM-005,staffing,0.586
1,ATM-001,capacity,0.549


TOP_K = 3


,id,area,similarity
0,ATM-005,staffing,0.586
1,ATM-001,capacity,0.549
2,ATM-004,communications,0.424


TOP_K = 5


,id,area,similarity
0,ATM-005,staffing,0.586
1,ATM-001,capacity,0.549
2,ATM-004,communications,0.424
3,ATM-010,communications,0.328
4,ATM-007,capacity,0.323


TOP_K = 8


,id,area,similarity
0,ATM-005,staffing,0.586
1,ATM-001,capacity,0.549
2,ATM-004,communications,0.424
3,ATM-010,communications,0.328
4,ATM-007,capacity,0.323
5,ATM-009,runway,0.302
6,ATM-014,coordination,0.290
7,ATM-008,weather,0.279


Un `top_k` pequeño puede dejar fuera información relevante. Uno demasiado grande puede introducir ruido, aumentar tokens/coste y distraer al modelo.

# 14. RAG es un patrón arquitectónico

Nuestro RAG usa:

```text
SentenceTransformer
        ↓
cosine similarity
        ↓
top-k
        ↓
prompt
        ↓
OpenAI Responses API
```

Pero cada componente puede sustituirse: búsqueda léxica o híbrida, vector database, reranker, otro modelo de embeddings, otro LLM, structured outputs, etc.

# 15. Vector stores y chunking

Con 15 documentos podemos comparar todos los vectores en memoria. Con millones de fragmentos usaríamos un índice/vector store.

También tendremos que resolver **chunking** cuando trabajemos con documentos reales:

```text
documento grande → chunks → embeddings → índice
```

Chunks demasiado grandes mezclan temas; demasiado pequeños pueden perder contexto.

# 16. Conexión con nuestro futuro proyecto de pruebas

Podemos trasladar ya la arquitectura:

```text
PASO DE PRUEBA EN LENGUAJE NATURAL
              │
              ▼
      recuperar documentación
      ejemplos / restricciones
              │
              ▼
             LLM
              │
              ▼
   interpretar qué acción realizar
              │
              ▼
       TOOL CALLING / MCP
              │
              ▼
        ejecutar acción real
```

RAG aporta **conocimiento**. MCP/tools aportarán **capacidad de actuar**.

# 17. Ejercicios

1. Añade cinco documentos ATM al corpus y recalcula embeddings.
2. Crea dos preguntas cuya respuesta necesite combinar dos documentos.
3. Prueba `top_k = 1`, `3` y `5`.
4. Crea una pregunta que el corpus no pueda responder.
5. Comprueba qué ocurre con y sin threshold.
6. Añade una pregunta y su documento esperado a `retrieval_eval`.
7. Modifica deliberadamente un documento para que contradiga a otro y observa la respuesta.

# 18. Preguntas de comprobación

Al terminar deberías poder explicar:

1. ¿Qué significa RAG?
2. ¿Qué parte corresponde a retrieval?
3. ¿Qué parte corresponde a generation?
4. ¿Por qué calculamos embeddings del corpus antes de recibir preguntas?
5. ¿Qué es `top_k`?
6. ¿Por qué el documento más similar no tiene necesariamente la respuesta?
7. ¿Por qué retrieval y generation deben evaluarse por separado?
8. ¿Qué ocurre si retrieval falla pero el LLM funciona perfectamente?
9. ¿Qué ocurre si retrieval es correcto pero generation falla?
10. ¿Por qué un threshold de similitud no es universal?
11. ¿Qué es Hit Rate@K?
12. ¿Qué papel tiene el chunking?
13. ¿Cuándo necesitamos un vector store?
14. ¿RAG modifica los parámetros del LLM?
15. ¿Qué diferencia existe entre RAG y fine-tuning?
16. ¿Qué aporta RAG al futuro proyecto de pruebas?
17. ¿Qué capacidad falta para que el LLM pueda actuar sobre un sistema real?

# 19. Siguiente paso — Tool Calling

Nuestro recorrido queda así:

```text
ML FUNDAMENTAL       ✓
EMBEDDINGS           ✓
SEMANTIC SEARCH      ✓
LLM                  ✓
PROMPT + CONTEXT     ✓
RAG                  ✓
        │
        ▼
TOOL CALLING
        │
        ▼
MCP
        │
        ▼
AGENTES
```

Ahora el sistema puede **buscar información y responder**. El siguiente paso será hacer que pueda **decidir utilizar una función externa, recibir su resultado y continuar**.